# 03 — Training and Evaluation
**Skin Lesion Classification in Thermal Images**

This notebook trains and evaluates all classifiers — custom CNN, transfer-learning CNN (ResNet/EfficientNet), SVM, and Random Forest — then compares their performance using accuracy, F1-score, AUC-ROC, confusion matrices, and learning curves.

## 1. Dataset e DataLoaders

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import torch

from src.dataset import (
    CLASSES,
    _collect_samples,
    _patient_split,
    build_dataloaders,
    compute_mean_std,
)

DATA_ROOT = Path("../data/processed/TR_heatSkin")

In [ ]:
# Distribuição de splits por paciente e por classe
samples = _collect_samples(DATA_ROOT)
splits = _patient_split(samples)

print(f"{'Split':<8} {'Frames':>8} {'Pacientes':>10} {'Healthy':>9} {'Sick':>7}")
print("-" * 46)
for name, split_samples in splits.items():
    n_healthy = sum(1 for s in split_samples if s["label"] == 0)
    n_sick    = sum(1 for s in split_samples if s["label"] == 1)
    n_patients = len(set(s["patient_id"] for s in split_samples))
    print(f"{name:<8} {len(split_samples):>8,} {n_patients:>10} {n_healthy:>9,} {n_sick:>7,}")

In [ ]:
# Calcula média e desvio padrão reais do treino (pode demorar ~1 min)
mean, std = compute_mean_std(splits["train"])
print(f"Média treino:  {mean:.4f}")
print(f"Desvio treino: {std:.4f}")

In [ ]:
# Monta os DataLoaders com mean/std calculados acima
train_loader, val_loader, test_loader = build_dataloaders(
    DATA_ROOT,
    batch_size=32,
    num_workers=0,
    mean=mean,
    std=std,
)

# Verifica shape e intervalo de valores de um batch
images, labels = next(iter(train_loader))
print(f"Shape do batch:  {tuple(images.shape)}")   # esperado: (32, 1, 224, 224)
print(f"Dtype:           {images.dtype}")
print(f"Valores min/max: {images.min():.3f} / {images.max():.3f}")
print(f"Labels únicos:   {labels.unique().tolist()}")

In [ ]:
# Visualiza 8 frames do batch com seus labels
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle("Amostras do batch de treino (após normalização → desnormalizado para exibição)", fontsize=11)

for i, ax in enumerate(axes.flat):
    img = images[i, 0]  # canal único
    img_display = img * std + mean  # desnormaliza para [0, 1]
    ax.imshow(img_display.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(CLASSES[labels[i].item()], fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()